# 05 — Business Impact and Final Hold-out Evaluation

Retrain the already selected model with the frozen training procedure, apply the already selected validation threshold, and evaluate the held-out test set **once**. Business-cost scenarios are illustrative assumptions, not bank accounting facts.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.business_metrics import BusinessCostAssumptions, cost_sensitivity_table, expected_cost
from src.data import load_dataset, split_dataset
from src.evaluation import classification_metrics
from src.models import FraudMLP, build_logistic_regression, build_random_forest, build_xgboost
from src.preprocessing import build_linear_preprocessor, build_tree_preprocessor

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FIGURES = ROOT / 'outputs' / 'figures'
METRICS = ROOT / 'outputs' / 'metrics'
REPORT = ROOT / 'report'
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

In [ ]:
selection = json.loads((METRICS / 'model_selection.json').read_text())
selected_model_name = selection['selected_model']
frozen_threshold = float(selection['selected_threshold'])

transactions = load_dataset(ROOT / 'data' / 'creditcard.csv')
X_train, X_validation, X_test, y_train, y_validation, y_test = split_dataset(transactions, random_state=SEED)
feature_names = X_train.columns.tolist()

selected_model_name, frozen_threshold

In [ ]:
def train_neural_network(X_train_array, y_train_series, X_validation_array, y_validation_series):
    class_counts = y_train_series.value_counts()
    positive_weight = float(class_counts.loc[0] / class_counts.loc[1])
    torch.manual_seed(SEED)
    model = FraudMLP(input_dim=X_train_array.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([positive_weight], dtype=torch.float32))

    train_features = torch.tensor(X_train_array, dtype=torch.float32)
    train_labels = torch.tensor(y_train_series.to_numpy(), dtype=torch.float32)
    validation_features = torch.tensor(X_validation_array, dtype=torch.float32)
    loader = DataLoader(TensorDataset(train_features, train_labels), batch_size=4096, shuffle=True)

    best_state = deepcopy(model.state_dict())
    best_pr_auc = -np.inf
    epochs_without_improvement = 0

    for _ in range(12):
        model.train()
        for batch_features, batch_labels in loader:
            optimizer.zero_grad()
            loss = loss_function(model(batch_features), batch_labels)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_scores = torch.sigmoid(model(validation_features)).numpy()
        validation_pr_auc = classification_metrics(y_validation_series, validation_scores)['pr_auc']

        if validation_pr_auc > best_pr_auc + 1e-5:
            best_pr_auc = validation_pr_auc
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= 3:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model

In [ ]:
linear_preprocessor = build_linear_preprocessor(feature_names)
tree_preprocessor = build_tree_preprocessor(feature_names)
X_train_linear = linear_preprocessor.fit_transform(X_train)
X_validation_linear = linear_preprocessor.transform(X_validation)
X_test_linear = linear_preprocessor.transform(X_test)
X_train_tree = tree_preprocessor.fit_transform(X_train)
X_validation_tree = tree_preprocessor.transform(X_validation)
X_test_tree = tree_preprocessor.transform(X_test)

if selected_model_name == 'Logistic Regression':
    final_model = build_logistic_regression(SEED)
    final_model.fit(X_train_linear, y_train)
    validation_scores = final_model.predict_proba(X_validation_linear)[:, 1]
    test_scores = final_model.predict_proba(X_test_linear)[:, 1]
elif selected_model_name == 'Random Forest':
    final_model = build_random_forest(SEED)
    final_model.fit(X_train_tree, y_train)
    validation_scores = final_model.predict_proba(X_validation_tree)[:, 1]
    test_scores = final_model.predict_proba(X_test_tree)[:, 1]
elif selected_model_name == 'XGBoost':
    class_counts = y_train.value_counts()
    positive_weight = float(class_counts.loc[0] / class_counts.loc[1])
    final_model = build_xgboost(SEED)
    final_model.set_params(scale_pos_weight=positive_weight)
    final_model.fit(X_train_tree, y_train)
    validation_scores = final_model.predict_proba(X_validation_tree)[:, 1]
    test_scores = final_model.predict_proba(X_test_tree)[:, 1]
elif selected_model_name == 'Neural Network':
    final_model = train_neural_network(X_train_linear, y_train, X_validation_linear, y_validation)
    with torch.no_grad():
        validation_scores = torch.sigmoid(final_model(torch.tensor(X_validation_linear, dtype=torch.float32))).numpy()
        test_scores = torch.sigmoid(final_model(torch.tensor(X_test_linear, dtype=torch.float32))).numpy()
else:
    raise ValueError(f'Unsupported selected model: {selected_model_name}')

In [ ]:
final_test_metrics = classification_metrics(y_test, test_scores, threshold=frozen_threshold)
final_test_result = {
    'model': selected_model_name,
    'threshold': frozen_threshold,
    'threshold_source': 'validation maximum F1',
    'primary_selection_metric': selection['primary_metric'],
    **final_test_metrics,
}
final_test_result

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
scenarios = {
    'low_review_cost': BusinessCostAssumptions(false_positive_review_cost=2.0),
    'base': BusinessCostAssumptions(false_positive_review_cost=5.0),
    'high_review_cost': BusinessCostAssumptions(false_positive_review_cost=20.0),
}

sensitivity_frames = []
for scenario_name, assumptions in scenarios.items():
    table = cost_sensitivity_table(
        y_validation, validation_scores, X_validation['Amount'].to_numpy(), thresholds, assumptions
    )
    table.insert(0, 'scenario', scenario_name)
    sensitivity_frames.append(table)
business_sensitivity = pd.concat(sensitivity_frames, ignore_index=True)
business_sensitivity.groupby('scenario').apply(lambda frame: frame.loc[frame['net_cost'].idxmin(), ['threshold', 'net_cost']], include_groups=False)

In [ ]:
base_assumptions = scenarios['base']
final_test_cost = expected_cost(
    y_test, test_scores, X_test['Amount'].to_numpy(), frozen_threshold, base_assumptions
)

figure, axis = plt.subplots(figsize=(9, 5))
for scenario_name, frame in business_sensitivity.groupby('scenario'):
    axis.plot(frame['threshold'], frame['net_cost'], label=scenario_name.replace('_', ' ').title())
axis.axvline(frozen_threshold, linestyle='--', linewidth=1, label='Frozen F1 threshold')
axis.set_xlabel('Decision threshold')
axis.set_ylabel('Illustrative net cost')
axis.set_title('Validation cost sensitivity under explicit assumptions')
axis.legend()
figure.tight_layout()
figure.savefig(FIGURES / '06_business_cost_sensitivity.png', dpi=180)
plt.show()

In [ ]:
with (METRICS / 'final_test_metrics.json').open('w') as file:
    json.dump(final_test_result, file, indent=2)
with (METRICS / 'final_test_business_cost.json').open('w') as file:
    json.dump({
        'assumptions': {
            'missed_fraud_cost_multiplier': base_assumptions.missed_fraud_cost_multiplier,
            'false_positive_review_cost': base_assumptions.false_positive_review_cost,
            'detected_fraud_recovery_rate': base_assumptions.detected_fraud_recovery_rate,
        },
        **final_test_cost,
    }, file, indent=2)
business_sensitivity.to_csv(METRICS / 'business_cost_sensitivity_validation.csv', index=False)

print(f"Final model: {selected_model_name}")
print(f"Frozen threshold: {frozen_threshold:.4f}")
print(f"Final hold-out PR-AUC: {final_test_metrics['pr_auc']:.4f}")
print('The cost figures are scenario outputs, not observed bank operating costs.')

## Interpretation

The hold-out result is an unbiased final check because the model family and decision threshold were fixed before test evaluation. The business layer does not claim that `Amount` equals the full loss borne by a bank: missed-fraud and manual-review assumptions are explicitly configurable, and sensitivity analysis shows how operational preferences can alter the economically preferred threshold without rewriting the statistical benchmark.